# Yap & Render: ISLRTC demo on Colab

Runs the `phase2-islrtc` branch with the ISLRTC dictionary read from Google Drive, and exposes `/demo` through a temporary public URL.
No GPU needed: use a plain CPU runtime. Run the cells top to bottom. After a runtime reset, run them all again (Drive files persist, everything else does not).

**Before you start, put these on Drive** (the manifest paths are relative to the dataset folder, so keep its layout unchanged):
- the dataset folder (the 13,665 clips)
- `manifest.json` and `approved_mappings.json` from the `ai-sign-language` indexer (`data/islrtc/`)

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings (edit the three Drive paths)

In [ ]:
import os, json

DRIVE_DATA_DIR = "/content/drive/MyDrive/islrtc"
DRIVE_MANIFEST = "/content/drive/MyDrive/islrtc_meta/manifest.json"
DRIVE_MAPPINGS = "/content/drive/MyDrive/islrtc_meta/approved_mappings.json"

COPY_TO_LOCAL = True   # copy the clips to Colab's local disk (much faster than reading Drive per request)
REPO_URL = "https://github.com/DAG-21/ai-sign-lang-yap-render.git"
BRANCH = "phase2-islrtc"

for p in (DRIVE_DATA_DIR, DRIVE_MANIFEST, DRIVE_MAPPINGS):
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

## 3. Check sizes and that the manifest matches the folder

In [ ]:
!du -sh "$DRIVE_DATA_DIR"
!df -h /content | tail -1

manifest = json.load(open(DRIVE_MANIFEST))
assets = manifest["assets"]
print(len(assets), "assets in manifest")
sample = [a for a in assets if a["status"] in ("source_label", "verified")][:200]
missing = [a["rel_path"] for a in sample if not os.path.exists(os.path.join(DRIVE_DATA_DIR, a["rel_path"]))]
print("sampled", len(sample), "clips, missing:", len(missing), missing[:3])

If `missing` is not 0, the folder layout on Drive differs from what the manifest expects. Fix that before continuing.
Compare the `du` size with the free space shown by `df`. If the data does not fit, set `COPY_TO_LOCAL = False` above and re-run from section 2.

## 4. Copy the dataset to local disk (skipped if COPY_TO_LOCAL is False)

In [ ]:
LOCAL_DATA_DIR = "/content/islrtc"
if COPY_TO_LOCAL:
    !rsync -a --info=progress2 "$DRIVE_DATA_DIR"/ "$LOCAL_DATA_DIR"/
    DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = DRIVE_DATA_DIR
print("ISLRTC_DATA_DIR =", DATA_DIR)

## 5. Install Node 22 (Next.js 16 needs Node 20.9 or newer) and FFmpeg check

In [ ]:
!node -v
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node -v && npm -v
!ffmpeg -version | head -1

## 6. Get the code

In [ ]:
%cd /content
!rm -rf yap-render
!git clone --branch $BRANCH $REPO_URL yap-render
%cd /content/yap-render
!git log --oneline -1
!npm ci --no-audit --no-fund

## 7. Run the tests (the 3 real-data end-to-end tests skip unless extra gloss files are set)

In [ ]:
!npm test

## 8. Build and start the server

In [ ]:
import subprocess, time, urllib.request

env = dict(os.environ,
           ISLRTC_DATA_DIR=DATA_DIR,
           ISLRTC_MANIFEST=DRIVE_MANIFEST,
           ISLRTC_MAPPINGS=DRIVE_MAPPINGS,
           ISLRTC_TRANSCODE="1",           # Chrome cannot decode the HEVC originals; convert requested clips to H.264
           ISLRTC_CACHE_DIR="/content/islrtc_cache",
           PORT="3000")

!cd /content/yap-render && npm run build 2>&1 | tail -15

server = subprocess.Popen("npm start", shell=True, cwd="/content/yap-render", env=env,
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:3000/demo", timeout=2); break
    except Exception:
        time.sleep(1)
print("server up" if server.poll() is None else "server exited, see /content/server.log")

## 9. Smoke test: dictionary loaded and a clip playable

In [ ]:
import requests

plan = requests.post("http://localhost:3000/api/sign-plan",
                     json={"tokens": ["HELLO", "STUDENT", "TODAY", "WORLD", "PROFESSOR", "FS:AMIT"]}).json()
print("dictionaryLoaded:", plan["dictionaryLoaded"])
for item in plan["plan"]:
    print(item["kind"], item.get("token") or item.get("label"))

clip = next((i for i in plan["plan"] if i["kind"] == "islrtc_video"), None)
if clip:
    r = requests.get(f"http://localhost:3000/api/islrtc/{clip['assetId']}?format=h264", headers={"Range": "bytes=0-1023"})
    print("clip status:", r.status_code, r.headers.get("content-type"))
else:
    print("no dictionary clip in this plan; check the plan output above and /content/server.log")

## 10. Public URL
Opens a temporary `trycloudflare.com` address (no account needed). Add `/demo` to it. The address changes each run and stops when the runtime stops. Share it only with people you trust.

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

import re
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: url = m.group(0); break
    time.sleep(1)
print((url + "/demo") if url else "no URL yet, see /content/tunnel.log")

## Notes
- Try `HELLO STUDENT TODAY WORLD PROFESSOR FS:AMIT THANK_YOU` in the demo. The first play of each dictionary clip waits 1 to 2 s for the H.264 conversion.
- Stop everything with `server.terminate(); tunnel.terminate()`.
- If something misbehaves, read `/content/server.log` and `/content/tunnel.log`.